# Métricas de clasificación: Precision, Recall y F1-score

<a href="https://colab.research.google.com/" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

En este cuaderno estudiaremos **Precision**, **Recall** y **F1-score** mediante un caso de control de calidad industrial. El modelo intentará identificar piezas defectuosas a partir de mediciones simuladas. La clase positiva será **defectuosa**.


## Objetivos del cuaderno

Al finalizar podrás:

- calcular Precision, Recall y F1-score desde una matriz de confusión;
- interpretar el efecto de falsos positivos y falsos negativos;
- comprobar los cálculos con `scikit-learn`;
- observar cómo cambia el desempeño al modificar el umbral;
- relacionar la métrica prioritaria con el costo del error.


## 1. Importar las herramientas

El cuaderno está preparado para Google Colab. No requiere subir archivos ni instalar paquetes adicionales.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, confusion_matrix,
    f1_score, precision_score, recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 2. Partir de una matriz de confusión conocida

Supongamos que un inspector automático revisó 100 piezas:

- **Clase positiva:** pieza defectuosa.
- **Clase negativa:** pieza correcta.
- VP = 30, FP = 10, FN = 5 y VN = 55.

Primero reconstruiremos estas etiquetas para conectar cada métrica con errores concretos.


In [ ]:
y_real_manual = np.array([0] * 65 + [1] * 35)
y_pred_manual = np.array([0] * 55 + [1] * 10 + [0] * 5 + [1] * 30)

matriz_manual = confusion_matrix(y_real_manual, y_pred_manual, labels=[0, 1])
matriz_manual


**Explicación del bloque:** este bloque continúa el paso anterior y muestra el resultado calculado o visualizado para poder interpretarlo antes de avanzar.\n

In [ ]:
ConfusionMatrixDisplay(
    confusion_matrix=matriz_manual,
    display_labels=['Correcta', 'Defectuosa'],
).plot(cmap='Oranges', colorbar=False)
plt.title('Matriz de confusión: inspección manual')
plt.xlabel('Clase predicha')
plt.ylabel('Clase real')
plt.show()


En la primera fila están las piezas realmente correctas: 55 se liberaron correctamente y 10 fueron enviadas a revisión por error. En la segunda fila están las piezas defectuosas: 30 fueron detectadas y 5 pasaron inadvertidas.


## 3. Calcular Precision

Precision responde:

> De todas las piezas que el sistema marcó como defectuosas, ¿cuántas realmente lo eran?

<div style="background-color:#fff3cd;border-left:6px solid #f0ad4e;padding:12px;font-size:1.15em"><b>Fórmula clave</b><br><br>$$Precision = \frac{VP}{VP + FP}$$<br><br><b>VP</b> = verdaderos positivos; <b>FP</b> = falsos positivos.</div>


In [ ]:
vn, fp, fn, vp = matriz_manual.ravel()
precision_manual = vp / (vp + fp)

pd.DataFrame({
    'VP': [vp], 'FP': [fp],
    'defectuosas_predichas': [vp + fp],
    'precision': [precision_manual],
}).round(3)


La inspección marcó 40 piezas como defectuosas y 30 lo eran. La Precision es 0.750: tres de cada cuatro alertas de defecto fueron correctas. Los falsos positivos reducen esta métrica porque generan revisiones innecesarias.


In [ ]:
precision_sklearn = precision_score(y_real_manual, y_pred_manual)
print(f'Precision manual:           {precision_manual:.3f}')
print(f'Precision con scikit-learn: {precision_sklearn:.3f}')


## 4. Calcular Recall

Recall responde:

> De todas las piezas que realmente eran defectuosas, ¿cuántas logró detectar el sistema?

<div style="background-color:#d9edf7;border-left:6px solid #31708f;padding:12px;font-size:1.15em"><b>Fórmula clave</b><br><br>$$Recall = \frac{VP}{VP + FN}$$<br><br><b>VP</b> = verdaderos positivos; <b>FN</b> = falsos negativos.</div>


In [ ]:
recall_manual = vp / (vp + fn)
pd.DataFrame({
    'VP': [vp], 'FN': [fn],
    'defectuosas_reales': [vp + fn],
    'recall': [recall_manual],
}).round(3)


Había 35 piezas defectuosas y se detectaron 30. El Recall es 0.857: se encontró aproximadamente el 85.7% de los defectos. Los 5 falsos negativos son piezas defectuosas que podrían llegar al cliente.


In [ ]:
recall_sklearn = recall_score(y_real_manual, y_pred_manual)
print(f'Recall manual:           {recall_manual:.3f}')
print(f'Recall con scikit-learn: {recall_sklearn:.3f}')


Precision mira las predicciones positivas; Recall mira los positivos reales. Por eso una inspección puede ser muy selectiva (Precision alta) o muy amplia (Recall alto), según el umbral utilizado.


## 5. Calcular el F1-score

F1-score combina Precision y Recall mediante su media armónica. Es alto solamente cuando ambas métricas son razonablemente altas.

<div style="background-color:#dff0d8;border-left:6px solid #3c763d;padding:12px;font-size:1.15em"><b>Fórmula clave</b><br><br>$$F1 = 2 \cdot \frac{Precision \cdot Recall}{Precision + Recall}$$<br><br>El F1-score penaliza el desequilibrio entre Precision y Recall.</div>


In [ ]:
f1_manual = 2 * precision_manual * recall_manual / (precision_manual + recall_manual)
f1_sklearn = f1_score(y_real_manual, y_pred_manual)

pd.DataFrame({
    'precision': [precision_manual], 'recall': [recall_manual],
    'F1_manual': [f1_manual], 'F1_sklearn': [f1_sklearn],
}).round(3)


El F1-score es aproximadamente 0.800. Resume el equilibrio entre una Precision de 0.750 y un Recall de 0.857; sin embargo, no sustituye la revisión de cada métrica por separado.


## 6. Un mismo F1-score puede ocultar comportamientos diferentes

Comparemos una inspección selectiva con una inspección amplia. Ambas pueden tener el mismo F1-score, pero implican costos distintos para la planta.


In [ ]:
escenarios_f1 = pd.DataFrame({
    'escenario': ['Inspección selectiva', 'Inspección amplia', 'Equilibrio'],
    'precision': [0.90, 0.60, 0.75],
    'recall': [0.60, 0.90, 0.75],
})
escenarios_f1['F1'] = (2 * escenarios_f1['precision'] * escenarios_f1['recall'] /
                       (escenarios_f1['precision'] + escenarios_f1['recall']))
escenarios_f1.round(3)


Las dos primeras situaciones tienen el mismo F1-score: una produce alertas confiables pero deja pasar defectos; la otra encuentra más defectos pero genera más falsas alarmas. El F1-score sirve como resumen, no como sustituto del contexto operativo.


## 7. Preparar un problema de clasificación realista

Ahora generaremos datos sintéticos de control de calidad. Cada fila representa una pieza y sus variables simuladas son mediciones de dimensiones, vibración y temperatura. La clase positiva (1) significa **defectuosa**. Se usará una proporción menor de defectos para representar un escenario de inspección industrial.


In [ ]:
X, y = make_classification(
    n_samples=1200, n_features=8, n_informative=5, n_redundant=1,
    n_clusters_per_class=2, weights=[0.86, 0.14],
    class_sep=0.9, flip_y=0.02, random_state=RANDOM_STATE,
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE,
)
print(f'Piezas totales: {len(y):,}')
print(f'Defectuosas: {y.sum():,} ({y.mean():.1%})')
print(f'Tamaño de prueba: {len(y_test):,}')


La división es estratificada para conservar una proporción semejante de piezas defectuosas en entrenamiento y prueba. Fijamos una semilla para que el ejemplo sea reproducible en Colab.


In [ ]:
modelo = Pipeline([
    ('escalador', StandardScaler()),
    ('logistica', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
modelo.fit(X_train, y_train)
print('Modelo entrenado.')


Usaremos regresión logística como clasificador didáctico. El objetivo no es optimizar el algoritmo, sino disponer de probabilidades para estudiar cómo el umbral modifica los errores y las métricas.


## 8. Evaluar el modelo con el umbral 0.5

El umbral habitual convierte una probabilidad de defecto de al menos 0.5 en la clase positiva. Conservamos Accuracy como referencia, pero el foco está en Precision, Recall y F1-score.


In [ ]:
prob_defecto = modelo.predict_proba(X_test)[:, 1]
pred_umbral_05 = (prob_defecto >= 0.5).astype(int)

metricas_05 = pd.DataFrame({
    'métrica': ['Accuracy', 'Precision', 'Recall', 'F1-score'],
    'valor': [
        accuracy_score(y_test, pred_umbral_05),
        precision_score(y_test, pred_umbral_05, zero_division=0),
        recall_score(y_test, pred_umbral_05, zero_division=0),
        f1_score(y_test, pred_umbral_05, zero_division=0),
    ],
})
metricas_05.round(3)


**Explicación del bloque:** este bloque continúa el paso anterior y muestra el resultado calculado o visualizado para poder interpretarlo antes de avanzar.\n

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, pred_umbral_05, labels=[0, 1],
    display_labels=['Correcta', 'Defectuosa'],
    cmap='Oranges', colorbar=False,
)
plt.title('Matriz de confusión con umbral 0.5')
plt.xlabel('Clase predicha')
plt.ylabel('Clase real')
plt.show()


La matriz permite explicar las métricas: los falsos positivos afectan Precision porque son alertas de piezas correctas; los falsos negativos afectan Recall porque son defectos que no se detectaron.


## 9. Observar el efecto del umbral

El modelo produce probabilidades. Al subir el umbral se vuelve más exigente para declarar una pieza defectuosa; al bajarlo, aumenta la cobertura esperada de defectos. Compararemos varios umbrales sin declarar todavía un ganador definitivo.


In [ ]:
resultados_umbrales = []
for umbral in [0.10, 0.25, 0.40, 0.50, 0.70, 0.90]:
    predicciones = (prob_defecto >= umbral).astype(int)
    vn_u, fp_u, fn_u, vp_u = confusion_matrix(y_test, predicciones, labels=[0, 1]).ravel()
    resultados_umbrales.append({
        'umbral': umbral, 'TN': vn_u, 'FP': fp_u, 'FN': fn_u, 'TP': vp_u,
        'precision': precision_score(y_test, predicciones, zero_division=0),
        'recall': recall_score(y_test, predicciones, zero_division=0),
        'F1': f1_score(y_test, predicciones, zero_division=0),
    })
tabla_umbrales = pd.DataFrame(resultados_umbrales)
tabla_umbrales.round(3)


**Explicación del bloque:** este bloque continúa el paso anterior y muestra el resultado calculado o visualizado para poder interpretarlo antes de avanzar.\n

In [ ]:
plt.figure(figsize=(8, 5))
for metrica in ['precision', 'recall', 'F1']:
    plt.plot(tabla_umbrales['umbral'], tabla_umbrales[metrica], marker='o', label=metrica)
plt.title('Precision, Recall y F1 según el umbral')
plt.xlabel('Umbral para declarar defecto')
plt.ylabel('Valor de la métrica')
plt.ylim(0, 1.05)
plt.grid(alpha=0.25)
plt.legend()
plt.show()


La curva muestra el intercambio habitual: umbrales bajos suelen aumentar Recall y las alertas; umbrales altos suelen aumentar Precision, aunque pueden dejar pasar más defectos. El mejor umbral depende del costo de cada tipo de error.


## 10. Comparar los errores de cada umbral

Veremos las matrices de un umbral bajo, uno intermedio y uno alto para conectar las cifras con piezas concretas.


In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(14, 4))
for eje, umbral in zip(ejes, [0.25, 0.50, 0.90]):
    predicciones = (prob_defecto >= umbral).astype(int)
    ConfusionMatrixDisplay.from_predictions(
        y_test, predicciones, labels=[0, 1],
        display_labels=['Correcta', 'Defectuosa'], cmap='Oranges',
        colorbar=False, ax=eje,
    )
    eje.set_title(f'Umbral {umbral:.2f}')
plt.tight_layout()
plt.show()


Al comparar las matrices, un umbral bajo prioriza no dejar pasar defectos, pero puede generar más falsos positivos. Un umbral alto reduce falsas alarmas, pero puede aumentar falsos negativos. Esa es la conexión entre decisiones operativas y métricas.


## 11. Priorizar Precision, Recall o equilibrio

Podemos resumir tres criterios: **priorizar Recall** cuando un defecto no detectado es muy costoso; **priorizar Precision** cuando cada revisión adicional es costosa; y **buscar equilibrio** cuando ambos errores importan de forma semejante.


In [ ]:
resumen_decisiones = tabla_umbrales[tabla_umbrales['umbral'].isin([0.25, 0.50, 0.90])].copy()
resumen_decisiones.insert(0, 'criterio_ilustrativo', ['Mayor cobertura', 'Referencia', 'Mayor exigencia'])
resumen_decisiones[['criterio_ilustrativo', 'umbral', 'FP', 'FN', 'precision', 'recall', 'F1']].round(3)


**Explicación del bloque:** este bloque continúa el paso anterior y muestra el resultado calculado o visualizado para poder interpretarlo antes de avanzar.\n

In [ ]:
fila_mejor_f1 = tabla_umbrales.loc[tabla_umbrales['F1'].idxmax()]
print(f"Mayor F1 entre los umbrales probados: {fila_mejor_f1['F1']:.3f}")
print(f"Umbral correspondiente: {fila_mejor_f1['umbral']:.2f}")
print(f"Precision: {fila_mejor_f1['precision']:.3f}")
print(f"Recall: {fila_mejor_f1['recall']:.3f}")


El umbral con mayor F1 solo es el mejor equilibrio dentro de los valores probados y de este conjunto de prueba. No demuestra que sea óptimo para producción: primero habría que cuantificar costos, validar con datos nuevos y vigilar el desempeño por lote o periodo.


## 12. Elegir la métrica según el contexto

| Situación | Error especialmente relevante | Métrica a observar |
|---|---|---|
| Liberar una pieza defectuosa al cliente | Falso negativo | Recall |
| Enviar piezas correctas a revisión manual | Falso positivo | Precision |
| Ambos errores importan de forma semejante | Balance entre ambos | F1-score |

En un proyecto real también conviene revisar la matriz de confusión, los costos por error y las métricas por segmento.


## 13. Una pequeña exploración

Modifica la lista de umbrales y observa cómo cambian las métricas. Evita interpretar el mayor F1 como una respuesta automática: revisa también FP, FN, Precision y Recall.


In [ ]:
umbrales_actividad = [0.15, 0.35, 0.55, 0.75, 0.95]
resultados_actividad = []
for umbral in umbrales_actividad:
    predicciones = (prob_defecto >= umbral).astype(int)
    resultados_actividad.append({
        'umbral': umbral,
        'precision': precision_score(y_test, predicciones, zero_division=0),
        'recall': recall_score(y_test, predicciones, zero_division=0),
        'F1': f1_score(y_test, predicciones, zero_division=0),
    })
pd.DataFrame(resultados_actividad).round(3)


Preguntas para explorar:

- ¿qué umbral produce menos falsos positivos?
- ¿cuál deja menos defectos sin detectar?
- ¿en cuál las alertas son más confiables?
- ¿qué costo tendría para la planta cada tipo de error?


## Cierre

Precision evalúa la confiabilidad de las predicciones positivas; Recall evalúa la capacidad para encontrar los positivos reales; F1-score resume el equilibrio entre ambas. En detección de defectos, cambiar el umbral cambia directamente el número de piezas enviadas a revisión y el número de defectos que podrían pasar inadvertidos.


## Para pensar

1. ¿Por qué los falsos positivos reducen Precision?
2. ¿Por qué los falsos negativos reducen Recall?
3. ¿Puede una inspección tener Precision alta y Recall bajo? ¿Cómo se vería en la matriz?
4. ¿Por qué un mismo F1-score puede corresponder a decisiones operativas diferentes?
5. Si liberar un defecto cuesta diez veces más que revisar una pieza correcta, ¿qué métrica priorizarías y por qué?
